In [4]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from time import time_ns
from typing import Dict, List, Optional, Tuple


class Side(Enum):
    BID = 0
    ASK = 1

    def opposite(self) -> "Side":
        return Side.ASK if self is Side.BID else Side.BID


class OrderType(Enum):
    LIMIT = "Limit"
    MARKET = "Market"
    IOC = "IOC"
    FOK = "FOK"
    STOP_LIMIT = "StopLimit"
    STOP_MARKET = "StopMarket"


@dataclass
class Order:
    id: str
    side: Side
    price: int
    size: int
    remaining: int
    timestamp_ns: int = 0
    order_type: OrderType = OrderType.LIMIT
    asset: int = 0
    stop_price: Optional[int] = None


@dataclass
class Fill:
    taker_id: str
    maker_id: str
    price: int
    size: int
    taker_side: Side


@dataclass
class OrderBook:
    bids: Dict[int, List[str]] = field(default_factory=dict)
    asks: Dict[int, List[str]] = field(default_factory=dict)
    orders: Dict[str, Order] = field(default_factory=dict)
    stop_orders: Dict[str, Order] = field(default_factory=dict)
    stop_bids: Dict[int, List[str]] = field(default_factory=dict)
    stop_asks: Dict[int, List[str]] = field(default_factory=dict)
    last_price: int = 0
    fills: List[Fill] = field(default_factory=list)

    @classmethod
    def new(cls) -> "OrderBook":
        return cls()

    def place(self, order: Order) -> List[Fill]:
        prev_fills = len(self.fills)
        prev_count = self.order_count()

        if order.order_type is OrderType.MARKET:
            self._match_market(order)
        elif order.order_type is OrderType.LIMIT:
            self._match_limit(order)
        elif order.order_type is OrderType.IOC:
            self._match_against_book(Order(**{**order.__dict__, "remaining": order.size}), False)
        elif order.order_type is OrderType.FOK:
            if order.size <= self._liquidity_at_price(order.side.opposite(), order.price):
                self._match_limit(Order(**{**order.__dict__, "remaining": order.size}))
        elif order.order_type in (OrderType.STOP_LIMIT, OrderType.STOP_MARKET):
            if order.stop_price is None:
                raise ValueError("stop orders require stop_price")
            self._add_stop_order(order, order.stop_price)
        else:
            raise ValueError(f"unsupported order type: {order.order_type}")

        return self.fills[prev_fills:]

    def restore_order(self, id: str, side: Side, price: int, size: int) -> None:
        restored = Order(
            id=id,
            side=side,
            price=price,
            size=size,
            remaining=size,
            timestamp_ns=0,
            order_type=OrderType.LIMIT,
            asset=0,
        )
        book = self.bids if side is Side.BID else self.asks
        book.setdefault(price, []).insert(0, id)
        self.orders[id] = restored

    def cancel(self, id: str) -> bool:
        order = self.orders.pop(id, None)
        if order is None:
            return False
        self._remove_from_level(order.side, order.price, id)
        return True

    def cancel_stop(self, id: str) -> bool:
        order = self.stop_orders.pop(id, None)
        if order is None:
            return False
        if order.stop_price is None:
            return False
        book = self.stop_bids if order.side is Side.BID else self.stop_asks
        ids = book.get(order.stop_price)
        if ids is not None:
            ids[:] = [x for x in ids if x != id]
            if not ids:
                book.pop(order.stop_price, None)
        return True

    def best_bid(self) -> Optional[Tuple[int, int]]:
        if not self.bids:
            return None
        price = max(self.bids)
        queue = self.bids[price]
        if not queue:
            return None
        order = self.orders.get(queue[0])
        return None if order is None else (price, order.remaining)

    def best_ask(self) -> Optional[Tuple[int, int]]:
        if not self.asks:
            return None
        price = min(self.asks)
        queue = self.asks[price]
        if not queue:
            return None
        order = self.orders.get(queue[0])
        return None if order is None else (price, order.remaining)

    def spread(self) -> Optional[int]:
        bid = self.best_bid()
        ask = self.best_ask()
        if bid is None or ask is None:
            return None
        return max(0, ask[0] - bid[0])

    def order_count(self) -> int:
        return len(self.orders)

    def depth(self, side: Side, levels: int) -> List[Tuple[int, int, int]]:
        book = self.bids if side is Side.BID else self.asks
        prices = sorted(book.keys(), reverse=(side is Side.BID))
        out: List[Tuple[int, int, int]] = []
        for price in prices[:levels]:
            queue = book[price]
            total = sum(self.orders[id].remaining for id in queue if id in self.orders)
            out.append((price, total, len(queue)))
        return out

    def _match_market(self, order: Order) -> None:
        while order.remaining > 0:
            best = self._pop_best(order.side.opposite())
            if best is None:
                break
            fill_size = min(order.remaining, best.remaining)
            self._record_fill(order, best, best.price, fill_size)
            order.remaining -= fill_size
            new_remaining = best.remaining - fill_size
            if new_remaining > 0:
                self._add_to_book(Order(**{**best.__dict__, "remaining": new_remaining}))
            else:
                self.orders.pop(best.id, None)

    def _match_limit(self, order: Order) -> None:
        self._match_against_book(order, True)

    def _match_against_book(self, order: Order, rest: bool) -> None:
        while order.remaining > 0:
            best = self._peek_best(order.side.opposite())
            if best is None:
                break
            crosses = (best.price <= order.price) if order.side is Side.BID else (best.price >= order.price)
            if not crosses:
                break
            best = self._pop_best(order.side.opposite())
            if best is None:
                break
            fill_size = min(order.remaining, best.remaining)
            self._record_fill(order, best, best.price, fill_size)
            order.remaining -= fill_size
            new_remaining = best.remaining - fill_size
            if new_remaining > 0:
                self._add_to_book(Order(**{**best.__dict__, "remaining": new_remaining}))
            else:
                self.orders.pop(best.id, None)
        if order.remaining > 0 and rest:
            self._add_to_book(Order(**{**order.__dict__, "remaining": order.remaining}))

    def _add_stop_order(self, order: Order, stop_price: int) -> None:
        self.stop_orders[order.id] = order
        book = self.stop_bids if order.side is Side.BID else self.stop_asks
        book.setdefault(stop_price, []).append(order.id)

    def _check_stops(self) -> None:
        if self.last_price == 0:
            return

        p = self.last_price

        triggered_bids = [price for price in sorted(self.stop_bids) if price <= p]
        for stop_price in triggered_bids:
            ids = list(self.stop_bids.get(stop_price, []))
            for id in ids:
                order = self.stop_orders.pop(id, None)
                if order is None:
                    continue
                trigger = Order(**{**order.__dict__, "order_type": OrderType.MARKET, "remaining": order.size})
                self._match_market(trigger)
            self.stop_bids.pop(stop_price, None)

        triggered_asks = [price for price in sorted(self.stop_asks) if price >= p]
        for stop_price in triggered_asks:
            ids = list(self.stop_asks.get(stop_price, []))
            for id in ids:
                order = self.stop_orders.pop(id, None)
                if order is None:
                    continue
                trigger = Order(**{**order.__dict__, "order_type": OrderType.MARKET, "remaining": order.size})
                self._match_market(trigger)
            self.stop_asks.pop(stop_price, None)

    def _add_to_book(self, order: Order) -> None:
        if order.remaining <= 0:
            return
        book = self.bids if order.side is Side.BID else self.asks
        book.setdefault(order.price, []).append(order.id)
        self.orders[order.id] = order

    def _remove_from_level(self, side: Side, price: int, id: str) -> None:
        book = self.bids if side is Side.BID else self.asks
        queue = book.get(price)
        if queue is None:
            return
        queue[:] = [x for x in queue if x != id]
        if not queue:
            book.pop(price, None)

    def _peek_best(self, side: Side) -> Optional[Order]:
        book = self.bids if side is Side.BID else self.asks
        if not book:
            return None
        price = max(book) if side is Side.BID else min(book)
        queue = book.get(price, [])
        if not queue:
            return None
        order = self.orders.get(queue[0])
        return None if order is None else Order(**order.__dict__)

    def _pop_best(self, side: Side) -> Optional[Order]:
        book = self.bids if side is Side.BID else self.asks
        if not book:
            return None
        price = max(book) if side is Side.BID else min(book)
        queue = book.get(price)
        if not queue:
            return None
        order_id = queue.pop(0)
        if not queue:
            book.pop(price, None)
        order = self.orders.pop(order_id, None)
        return None if order is None else order

    def _record_fill(self, taker: Order, maker: Order, price: int, size: int) -> None:
        self.last_price = price
        self.fills.append(Fill(taker.id, maker.id, price, size, taker.side))
        self._check_stops()

    def _liquidity_at_price(self, side: Side, price: int) -> int:
        book = self.bids if side is Side.BID else self.asks
        prices = sorted(book.keys(), reverse=(side is Side.BID))
        total = 0
        for level_price in prices:
            if (side is Side.BID and level_price < price) or (side is Side.ASK and level_price > price):
                break
            total += sum(self.orders[id].remaining for id in book[level_price] if id in self.orders)
        return total


In [5]:
# Quick scratchpad for the translated model.
book = OrderBook.new()
book.place(order("a1", Side.ASK, 100, 20, OrderType.LIMIT))
book.place(order("a2", Side.ASK, 101, 15, OrderType.LIMIT))
book.cancel("a2")
book.best_ask(), book.depth(Side.ASK, 3)


((100, 20), [(100, 20, 1)])

In [9]:
@dataclass
class MatchParams:
    match_price: int
    match_size: int


def now_nanos() -> int:
    return time_ns()


def order(id: str, side: Side, price: int, size: int, order_type: OrderType) -> Order:
    return Order(
        id=id,
        side=side,
        price=price,
        size=size,
        remaining=size,
        timestamp_ns=now_nanos(),
        order_type=order_type,
        asset=0,
    )


def is_bid(side: int) -> bool:
    return side in (0, 3)


def find_match(a, b) -> Optional[MatchParams]:
    if is_bid(a.side) == is_bid(b.side):
        return None

    buy, sell = (a, b) if is_bid(a.side) else (b, a)
    buy_price = sell.price if buy.price == 0 else buy.price
    sell_price = buy.price if sell.price == 0 else sell.price

    if buy_price < sell_price:
        return None

    return MatchParams(
        match_price=(buy_price + sell_price) // 2,
        match_size=min(buy.size, sell.size),
    )


book = OrderBook.new()
book.place(order("a1", Side.ASK, 100, 20, OrderType.LIMIT))
book.place(order("b1", Side.BID, 101, 10, OrderType.LIMIT))
book.best_bid(), book.best_ask(), book.fills


(None,
 (100, 10),
 [Fill(taker_id='b1', maker_id='a1', price=100, size=10, taker_side=<Side.BID: 0>)])